# Notebook 03 – Avaliação e Detecção de Drift em Produção

## Objetivos

- Avaliar todas as métricas de drift de forma consolidada
- Simular drift progressivo e em alta dimensionalidade
- Integrar detecção de drift em pipelines MLOps
- Explorar ferramentas de produção como Alibi Detect

**Referência:** Documento 04 – Métricas Avançadas para Detecção de Drift (Vídeo 4: Detecção de Drift Multivariado em Produção).
Seção de estado da arte sobre ferramentas de monitoramento e integração em produção.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Adiciona o diretório raiz ao path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_preprocessing import DataPreprocessor
from src.model import (
    PSICalculator, MMDCalculator, WassersteinCalculator,
    EnergyDistanceCalculator, DriftDetector, DriftResult
)
from src.evaluation import (
    calculate_metrics, plot_distributions, plot_scatter_drift,
    plot_metric_comparison, plot_correlation_heatmaps
)
from src.utils import load_model, save_metrics, setup_logging

logger = setup_logging()
print("Imports carregados com sucesso.")

## 1. Carregamento dos Dados e Detector

Carregamos o dataset sintético e o detector previamente treinado nos notebooks anteriores.
O `DataPreprocessor` gera ou carrega os dados de referência e atuais, simulando
uma mudança de distribuição entre períodos (Documento 04).

In [ ]:
# Gerar/carregar dados
preprocessor = DataPreprocessor(n_samples=5000, seed=42)
df = preprocessor.generate_dataset()
df = preprocessor.clean_data(df)
df = preprocessor.prepare_features(df, normalize=True)

# Separar períodos de referência e atual
df_ref, df_cur = preprocessor.split_data(df)

feature_cols = [c for c in df_ref.columns if c != "periodo"]
X_ref = df_ref[feature_cols].values
X_cur = df_cur[feature_cols].values

print(f"Período de referência: {X_ref.shape}")
print(f"Período atual: {X_cur.shape}")
print(f"Features: {feature_cols}")

# Tentar carregar detector salvo ou criar novo
try:
    detector = load_model()
    print("Detector carregado do disco.")
except FileNotFoundError:
    detector = DriftDetector(psi_threshold=0.25)
    detector.fit(X_ref)
    print("Novo detector criado e ajustado aos dados de referência.")

## 2. Avaliação Consolidada de Todas as Métricas

Conforme discutido no **Documento 04**, métricas univariadas como PSI analisam cada
feature individualmente, enquanto métricas multivariadas como MMD, Wasserstein e
Energy Distance capturam mudanças na estrutura conjunta dos dados.

Nesta seção, aplicamos todas as métricas simultaneamente e comparamos seus resultados.

In [ ]:
# Executar predição com todas as métricas
results = detector.predict(X_cur)

# Tabela formatada de resultados
print("=" * 75)
print(f"{"Métrica":<25} {"Estatística":>12} {"Threshold":>12} {"Drift?":>10}")
print("=" * 75)
for name, result in results.items():
    threshold_str = f"{result.threshold:.4f}" if result.threshold is not None else "N/A"
    drift_str = "SIM" if result.drift_detected else "NÃO"
    print(f"{result.metric_name:<25} {result.statistic:>12.4f} {threshold_str:>12} {drift_str:>10}")
print("=" * 75)

# Métricas consolidadas via evaluation module
metrics = calculate_metrics(X_ref, X_cur, feature_names=feature_cols)
print(f"\nResumo das métricas: {list(metrics.keys())}")

## 3. Visualização Comparativa

As visualizações abaixo permitem identificar o drift de forma intuitiva.
Conforme a **Figura 1 do Documento 04**, a comparação de distribuições e
matrizes de correlação revela mudanças que métricas univariadas podem não captar.

In [ ]:
# Distribuições de referência vs atual
fig1 = plot_distributions(X_ref, X_cur, feature_names=feature_cols)
plt.show()

# Scatter plot mostrando drift multivariado
fig2 = plot_scatter_drift(X_ref, X_cur, feature_x=0, feature_y=1,
                          labels=("Referência", "Atual"))
plt.show()

# Comparação de métricas
fig3 = plot_metric_comparison(metrics)
plt.show()

# Heatmaps de correlação
fig4 = plot_correlation_heatmaps(X_ref, X_cur, feature_names=feature_cols)
plt.show()

## 4. Simulação de Drift Progressivo

Para avaliar a sensibilidade de cada métrica, simulamos drift progressivo
variando a intensidade da perturbação nos dados. Conforme o **Documento 04**,
testes não supervisionados comparam janelas de dados sem necessidade de rótulos
[Cloudera FFL, 2021], permitindo detectar drift em diferentes graus de severidade.

In [ ]:
# Simular drift com intensidades crescentes
np.random.seed(42)
n_samples = 1000
n_features = X_ref.shape[1]
intensidades = np.linspace(0, 3, 15)

resultados_drift = []

for intensidade in intensidades:
    # Dados de referência estáveis
    ref = np.random.randn(n_samples, n_features)

    # Dados atuais com drift progressivo (deslocamento de média)
    cur = np.random.randn(n_samples, n_features) + intensidade * 0.5

    # Adicionar mudança de correlação progressiva
    if n_features >= 2:
        fator_corr = intensidade / 3.0
        cur[:, 1] = cur[:, 1] + fator_corr * cur[:, 0]

    # Calcular todas as métricas
    psi_calc = PSICalculator()
    mmd_calc = MMDCalculator()
    wass_calc = WassersteinCalculator()
    energy_calc = EnergyDistanceCalculator()

    psi_vals = psi_calc.calculate_multifeature(ref, cur)
    psi_medio = np.mean(psi_vals)
    mmd_val = mmd_calc.calculate(ref, cur)
    wass_vals = wass_calc.calculate_multifeature(ref, cur)
    wass_medio = np.mean(wass_vals)
    energy_val = energy_calc.calculate(ref, cur)

    resultados_drift.append({
        "intensidade": intensidade,
        "PSI (médio)": psi_medio,
        "MMD": mmd_val,
        "Wasserstein (médio)": wass_medio,
        "Energy Distance": energy_val
    })

df_drift = pd.DataFrame(resultados_drift)
print(df_drift.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metricas_plot = ["PSI (médio)", "MMD", "Wasserstein (médio)", "Energy Distance"]
cores = ["#e74c3c", "#2ecc71", "#3498db", "#9b59b6"]

for ax, metrica, cor in zip(axes.flat, metricas_plot, cores):
    ax.plot(df_drift["intensidade"], df_drift[metrica],
            marker="o", color=cor, linewidth=2)
    ax.set_title(metrica, fontsize=13, fontweight="bold")
    ax.set_xlabel("Intensidade do Drift")
    ax.set_ylabel("Valor da Métrica")
    ax.grid(True, alpha=0.3)

plt.suptitle("Sensibilidade das Métricas ao Drift Progressivo",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

print("Observação: MMD e Energy Distance capturam drift multivariado")
print("que o PSI pode não detectar em features isoladas.")

## 5. Detecção de Drift em Alta Dimensionalidade

O **Documento 04** destaca que a detecção de drift em dados de alta dimensionalidade
é um desafio significativo. A maldição da dimensionalidade pode diluir sinais de drift
em features individuais, mas métricas multivariadas como MMD e Energy Distance
permanecem eficazes ao capturar mudanças na distribuição conjunta.

Greco et al. (2024) propõem o uso de embeddings de deep learning para reduzir
a dimensionalidade antes de aplicar testes de drift, uma abordagem promissora
para dados de alta complexidade.

In [ ]:
# Gerar dados de alta dimensionalidade
np.random.seed(42)
n_samples_hd = 500
n_features_hd = 25

# Referência: features independentes
X_ref_hd = np.random.randn(n_samples_hd, n_features_hd)

# Atual: drift sutil em correlações (sem mudar marginais)
X_cur_hd = np.random.randn(n_samples_hd, n_features_hd)
for i in range(0, 10, 2):
    X_cur_hd[:, i + 1] = 0.6 * X_cur_hd[:, i] + 0.4 * X_cur_hd[:, i + 1]

# Aplicar todas as métricas
psi_calc = PSICalculator()
mmd_calc = MMDCalculator()
wass_calc = WassersteinCalculator()
energy_calc = EnergyDistanceCalculator()

psi_hd = np.mean(psi_calc.calculate_multifeature(X_ref_hd, X_cur_hd))
mmd_hd = mmd_calc.calculate(X_ref_hd, X_cur_hd)
wass_hd = np.mean(wass_calc.calculate_multifeature(X_ref_hd, X_cur_hd))
energy_hd = energy_calc.calculate(X_ref_hd, X_cur_hd)

print(f"Drift em Alta Dimensionalidade ({n_features_hd} features)")
print("=" * 55)
print(f"{"Métrica":<25} {"Valor":>12} {"Detecta Drift?":>15}")
print("-" * 55)
print(f"{"PSI (médio)":<25} {psi_hd:>12.4f} {"Provavelmente não":>15}")
print(f"{"MMD":<25} {mmd_hd:>12.4f} {"Sim":>15}")
print(f"{"Wasserstein (médio)":<25} {wass_hd:>12.4f} {"Parcialmente":>15}")
print(f"{"Energy Distance":<25} {energy_hd:>12.4f} {"Sim":>15}")
print("\nMétricas multivariadas detectam mudanças em correlação mesmo sem")
print("alteração nas distribuições marginais individuais.")

## 6. Monitoramento em Produção com Alibi Detect

O **Documento 04** apresenta ferramentas open-source para detecção de drift em produção:

- **Alibi Detect**: biblioteca robusta com suporte a MMD, LSDD e outros testes [Müller et al., 2024 — D3Bench]
- **Evidently AI**: monitoramento de modelos com dashboards interativos
- **NannyML**: detecção de drift e estimativa de performance sem ground truth

O LSDD (Least-Squares Density Difference) [Sugiyama et al., 2013] é uma alternativa
ao MMD que estima diretamente a diferença entre densidades.

> **Nota:** Esta seção é opcional e requer o pacote `alibi-detect` instalado.

In [ ]:
# Preparar dados como arrays numpy float32
X_ref_np = X_ref.astype(np.float32)
X_cur_np = X_cur.astype(np.float32)

try:
    from alibi_detect.cd import MMDDrift

    # Criar detector MMD com dados de referência
    cd = MMDDrift(X_ref_np, backend="numpy", p_val=0.05)

    # Executar predição nos dados atuais
    preds = cd.predict(X_cur_np)

    print("=== Alibi Detect - MMDDrift ===")
    print(f"Drift detectado: {preds['data']['is_drift']}")
    print(f"p-valor: {preds['data']['p_val']:.4f}")
    print(f"Estatística MMD: {preds['data']['distance']:.4f}")
    print(f"Threshold (p=0.05): {preds['data']['distance_threshold']:.4f}")

except ImportError:
    print("Alibi Detect não instalado. Instale com: pip install alibi-detect")
    print("Usando implementação própria (src.model.MMDCalculator) como alternativa.")

    # Fallback: usar implementação própria
    mmd_calc = MMDCalculator()
    mmd_result = mmd_calc.permutation_test(X_ref, X_cur)
    print(f"\nMMDCalculator (implementação própria):")
    print(f"  Estatística MMD: {mmd_result.statistic:.4f}")
    print(f"  Drift detectado: {mmd_result.drift_detected}")
    print(f"  Detalhes: {mmd_result.details}")

## 7. Integração em Pipeline MLOps

O **Documento 04** enfatiza a importância de integrar a detecção de drift em pipelines
de produção. Testes não supervisionados são particularmente valiosos porque não
requerem rótulos para funcionar — basta comparar janelas de dados ao longo do tempo
[Cloudera FFL, 2021].

Uma pipeline típica de monitoramento:
1. Coletar dados em janelas temporais (diária, semanal)
2. Comparar cada janela com os dados de referência
3. Alertar quando drift significativo é detectado
4. Acionar retreinamento automático se necessário

In [ ]:
def monitorar_drift_pipeline(X_referencia, dados_producao,
                              tamanho_janela=200, psi_threshold=0.25):
    """
    Simula monitoramento de drift em produção processando dados
    em janelas temporais.
    Conforme Documento 04: testes não supervisionados comparam
    janelas de dados sem necessidade de rótulos.
    """
    detector_pipeline = DriftDetector(psi_threshold=psi_threshold)
    detector_pipeline.fit(X_referencia)

    n_janelas = len(dados_producao) // tamanho_janela
    historico = []

    print(f"Monitorando {n_janelas} janelas de {tamanho_janela} amostras cada")
    print("=" * 65)

    for i in range(n_janelas):
        inicio = i * tamanho_janela
        fim = inicio + tamanho_janela
        janela = dados_producao[inicio:fim]

        resultados = detector_pipeline.predict(janela)
        n_alertas = sum(1 for r in resultados.values() if r.drift_detected)

        status = "ALERTA" if n_alertas >= 2 else "OK"
        historico.append({
            "janela": i + 1,
            "inicio": inicio,
            "fim": fim,
            "alertas": n_alertas,
            "status": status,
            "mmd": resultados["mmd"].statistic
        })

        print(f"Janela {i + 1:>3}: amostras [{inicio:>5}:{fim:>5}] | "
              f"Alertas: {n_alertas}/4 | Status: {status} | "
              f"MMD: {resultados['mmd'].statistic:.4f}")

    print("=" * 65)
    df_hist = pd.DataFrame(historico)
    total_alertas = (df_hist["status"] == "ALERTA").sum()
    print(f"\nResumo: {total_alertas}/{n_janelas} janelas com drift detectado")
    return df_hist


# Simular dados de produção com drift gradual
np.random.seed(42)
n_prod = 2000
n_feat = X_ref.shape[1]

# Primeira metade: sem drift; segunda metade: com drift
dados_prod_1 = np.random.randn(n_prod // 2, n_feat)
dados_prod_2 = np.random.randn(n_prod // 2, n_feat) + 1.0
dados_producao = np.vstack([dados_prod_1, dados_prod_2])

df_monitoramento = monitorar_drift_pipeline(X_ref, dados_producao)

## 8. Casos Reais e Tendências

O **Documento 04** apresenta casos reais que ilustram a importância do monitoramento:

- **Instacart (COVID-19):** Queda de acurácia de 93% para 61% durante a pandemia,
  quando padrões de compra mudaram drasticamente [Documento 04]
- **Filtros de spam:** Drift contínuo conforme spammers adaptam estratégias
  [Feldhans et al., 2021]
- **Deep learning para drift:** Uso de embeddings neurais para detecção de drift
  em dados complexos [Greco et al., 2024]
- **LSDD:** Estimativa direta de diferença de densidades como alternativa ao MMD
  [Sugiyama et al., 2013]

Müller et al. (2024) desenvolveram o D3Bench, benchmark que compara ferramentas
como Alibi Detect, Evidently AI e NannyML em cenários padronizados.

In [ ]:
# Tabela de casos reais com métricas apropriadas (Documento 04)
casos_reais = pd.DataFrame([
    {
        "Caso": "Instacart (COVID-19)",
        "Tipo de Drift": "Covariável (mudança de padrão de compra)",
        "Impacto": "Acurácia: 93% → 61%",
        "Métrica Recomendada": "MMD / Energy Distance",
        "Referência": "Documento 04"
    },
    {
        "Caso": "Filtro de Spam",
        "Tipo de Drift": "Gradual (adaptação de spammers)",
        "Impacto": "Degradação contínua de precision",
        "Métrica Recomendada": "PSI + Wasserstein (monitoramento contínuo)",
        "Referência": "Feldhans et al., 2021"
    },
    {
        "Caso": "Detecção via Deep Learning",
        "Tipo de Drift": "Alta dimensionalidade (embeddings)",
        "Impacto": "Detecção precoce em dados complexos",
        "Métrica Recomendada": "MMD sobre embeddings",
        "Referência": "Greco et al., 2024"
    },
    {
        "Caso": "LSDD para drift de densidade",
        "Tipo de Drift": "Mudança de distribuição",
        "Impacto": "Alternativa ao MMD com estimativa direta",
        "Métrica Recomendada": "LSDD (Least-Squares Density Difference)",
        "Referência": "Sugiyama et al., 2013"
    },
    {
        "Caso": "Benchmark D3Bench",
        "Tipo de Drift": "Múltiplos cenários padronizados",
        "Impacto": "Comparação de ferramentas open-source",
        "Métrica Recomendada": "Alibi Detect / Evidently / NannyML",
        "Referência": "Müller et al., 2024"
    }
])

print("Casos Reais de Data Drift e Métricas Recomendadas (Documento 04)")
print("=" * 90)
for _, row in casos_reais.iterrows():
    print(f"\n\U0001f4cc {row['Caso']}")
    print(f"   Tipo: {row['Tipo de Drift']}")
    print(f"   Impacto: {row['Impacto']}")
    print(f"   Métrica: {row['Métrica Recomendada']}")
    print(f"   Ref: {row['Referência']}")

## 9. Salvando Resultados

Persistimos as métricas finais e figuras para consulta posterior.
Os resultados são salvos no diretório `outputs/` conforme organização do projeto.

In [ ]:
# Criar diretórios de saída
output_dir = project_root / "outputs"
figures_dir = output_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# Salvar métricas em JSON
metrics_path = save_metrics(metrics)
print(f"Métricas salvas em: {metrics_path}")

# Salvar figuras
fig_dist = plot_distributions(
    X_ref, X_cur, feature_names=feature_cols,
    save_path=str(figures_dir / "distribuicoes.png")
)
fig_scatter = plot_scatter_drift(
    X_ref, X_cur, feature_x=0, feature_y=1,
    save_path=str(figures_dir / "scatter_drift.png")
)
fig_comp = plot_metric_comparison(
    metrics,
    save_path=str(figures_dir / "comparacao_metricas.png")
)
fig_corr = plot_correlation_heatmaps(
    X_ref, X_cur, feature_names=feature_cols,
    save_path=str(figures_dir / "correlacoes.png")
)
plt.close("all")

print(f"Figuras salvas em: {figures_dir}")
print("Notebook 03 finalizado com sucesso.")

## Resumo da Aula

### Recapitulação dos 3 Notebooks (Documento 04 – Vídeos 1 a 4)

**Notebook 01 – Introdução e Métricas Univariadas (Vídeos 1–2):**
- Conceitos fundamentais de data drift
- PSI e teste de Kolmogorov-Smirnov para detecção univariada
- Limitações: análise feature a feature não captura mudanças na estrutura conjunta

**Notebook 02 – Métricas Multivariadas (Vídeo 3):**
- MMD (Maximum Mean Discrepancy) com kernel RBF
- Wasserstein Distance e Energy Distance
- Demonstração de que drift em correlações passa despercebido pelo PSI

**Notebook 03 – Avaliação e Produção (Vídeo 4):**
- Avaliação consolidada de todas as métricas
- Simulação de drift progressivo e alta dimensionalidade
- Ferramentas de produção: Alibi Detect, Evidently AI, NannyML [Müller et al., 2024]
- Integração em pipelines MLOps com monitoramento por janelas temporais
- Casos reais: Instacart, filtros de spam, deep learning para drift

### Principais Conclusões (Documento 04)

1. **Testes univariados têm pontos cegos:** PSI analisa cada feature separadamente
   e não detecta mudanças em correlações entre features.

2. **Métricas multivariadas são essenciais:** MMD, Wasserstein e Energy Distance
   capturam drift sutil na distribuição conjunta dos dados.

3. **Testes não supervisionados são práticos:** Não requerem rótulos — basta
   comparar janelas de dados [Cloudera FFL, 2021].

4. **Ferramentas open-source facilitam a produção:** Alibi Detect, Evidently AI
   e NannyML oferecem implementações prontas e validadas [Müller et al., 2024].

5. **Monitoramento contínuo é crítico:** O caso Instacart (93% → 61%) demonstra
   que drift pode causar degradação severa e rápida de modelos em produção.

---

**Referências:**
- Documento 04 – Métricas Avançadas para Detecção de Drift
- Müller et al. (2024) – D3Bench: benchmark para ferramentas de drift
- Greco et al. (2024) – Deep learning para detecção de drift
- Sugiyama et al. (2013) – LSDD (Least-Squares Density Difference)
- Feldhans et al. (2021) – Drift em filtros de spam
- Cloudera FFL (2021) – Monitoramento não supervisionado de modelos